# SynBioCrow 2.2.0 — final release build
This notebook builds the final 2.2.0 release from one exact source commit after RC2 published-artifact acceptance passed. It introduces no intentional scientific algorithm changes beyond final version/release metadata.


In [ ]:
import os,sys,subprocess,shutil,json,hashlib
from pathlib import Path
from google.colab import drive

REPO_URL="https://github.com/theiman112860/SynBioCrow.git"
SOURCE_COMMIT="1e54f885120209746a385443b9390590d442dc4c"
ROOT=Path("/content/SynBioCrow_2_2_0_FINAL")
if ROOT.exists(): shutil.rmtree(ROOT)

def run(cmd,cwd=None,timeout=3600):
    print("$"," ".join(map(str,cmd)),flush=True)
    p=subprocess.run(list(map(str,cmd)),cwd=cwd,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,timeout=timeout)
    print(p.stdout,flush=True)
    if p.returncode:
        raise RuntimeError(f"command failed rc={p.returncode}: {cmd}")
    return p.stdout

run(["git","clone",REPO_URL,str(ROOT)])
run(["git","checkout",SOURCE_COMMIT],cwd=ROOT)
head=run(["git","rev-parse","HEAD"],cwd=ROOT).strip()
assert head==SOURCE_COMMIT,(head,SOURCE_COMMIT)
print("FINAL SOURCE COMMIT LOCK: PASS",head)


In [ ]:
# Install and verify final package metadata in fresh subprocesses.
run([sys.executable,"-m","pip","install","-q","-e",str(ROOT)])
run([sys.executable,"-c","import synbiocrow; print('synbiocrow import PASS',synbiocrow.__file__)"])
out=run([sys.executable,"-c","from importlib.metadata import version; print(version('synbiocrow'))"]).strip().splitlines()[-1]
assert out=="2.2.0",out
print("FINAL INSTALL/VERSION CHECK: PASS",out)


In [ ]:
# Full test suite.
run([sys.executable,"-m","pip","install","-q","pytest"])
run([sys.executable,"-m","pytest","-q","tests"],cwd=ROOT,timeout=1800)
print("FINAL TEST SUITE: PASS")


In [ ]:
# Validate final release metadata and RC2 acceptance provenance.
pyproject=(ROOT/"pyproject.toml").read_text()
readme=(ROOT/"README.md").read_text()
notes=(ROOT/"RELEASE_NOTES_2_2_0.md").read_text()
assert 'version = "2.2.0"' in pyproject
assert "# SynBioCrow 2.2.0" in readme
assert "2.2.0 Release Notes" in notes
for token in [
 "631bac03b9a930a29721b955d1a06515942df23c",
 "cacec275839f4fccb36f5c6732ab99d4c132c92c09e89c6ce3df86c86687dca5",
 "db4d601dc82768916f2a80190b3c45dde13c4a864c304f16636f10f937447237",
 "ba0b5ae9f11ba073a0e2f015b548051bfaa76f2483a5eac63b50c6d8ab293c73",
]:
    assert token in notes,token
print("FINAL RELEASE METADATA + RC2 ACCEPTANCE PROVENANCE: PASS")


In [ ]:
# Re-run M7 and M9 from the exact final source commit.
run([sys.executable,str(ROOT/"scripts/m7_release_readiness.py")],cwd=ROOT,timeout=1800)
run([sys.executable,str(ROOT/"scripts/build_m9_release.py"),"--check"],cwd=ROOT,timeout=1800)
print("FINAL M7 + M9 CHECK: PASS")


In [ ]:
# Build final reproducibility package to Google Drive.
drive.mount("/content/drive",force_remount=False)
OUT=Path("/content/drive/MyDrive/SynBioCrow/release/2.2.0")
OUT.mkdir(parents=True,exist_ok=True)
run([sys.executable,str(ROOT/"scripts/build_m9_release.py"),"--output-dir",str(OUT)],cwd=ROOT,timeout=1800)

def sha256(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for c in iter(lambda:f.read(1024*1024),b""): h.update(c)
    return h.hexdigest()

print("\nFINAL 2.2.0 OUTPUTS")
for p in sorted(OUT.glob("*")):
    print(p.name,p.stat().st_size,sha256(p))

summary=json.loads((OUT/"SynBioCrow_2.2.0_PACKAGE_SUMMARY.json").read_text())
manifest=json.loads((OUT/"SynBioCrow_2.2.0_release_manifest.json").read_text())
assert summary["git_commit"]==SOURCE_COMMIT,summary["git_commit"]
assert manifest["git_commit"]==SOURCE_COMMIT,manifest["git_commit"]
assert manifest["release"]=="2.2.0",manifest["release"]
assert manifest["release_kind"]=="final",manifest["release_kind"]
print("FINAL PACKAGE COMMIT MATCH: PASS",SOURCE_COMMIT)
print("SYNBIOCROW 2.2.0 FINALIZATION: PASS")
print("Next action: freeze/tag this exact commit as v2.2.0 and attach these three final M9 assets.")
